<a href="https://colab.research.google.com/github/harsh123-dev/EDA-Course-Project/blob/main/EDA_Phase_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA Course Project – Phase 1

## Exploratory Data Analysis on the BEPS Dataset

This notebook completes the required Phase 1 tasks:

1. Loading the dataset
2. Basic statistical analysis
3. Handling missing data
4. Data cleaning
5. Data transformation
6. Univariate analysis – minimum 3 visualizations
7. Bivariate analysis – minimum 3 visualizations
8. Multivariate analysis – minimum 3 visualizations

The analysis is descriptive and focuses on the structure, distributions, and relationships present in the dataset.


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 110


## 2. Loading the Dataset


In [ ]:
dataset_url = "https://raw.githubusercontent.com/salemprakash/EDA/main/Data/BEPS.csv"

df = pd.read_csv(dataset_url)

print("Dataset loaded successfully.")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())


## 3. Dataset Overview


In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDataset information:")
df.info()

print("\nFirst five rows:")
display(df.head())


## 4. Basic Statistical Analysis


In [ ]:
print("Numerical descriptive statistics:")
display(df.describe(include=[np.number]).T)

print("Categorical descriptive statistics:")
display(df.select_dtypes(include=["object", "category", "bool"]).describe().T)

print("Number of unique values in each column:")
display(df.nunique().sort_values(ascending=False).to_frame("Unique Values"))


## 5. Missing Data Analysis


In [ ]:
missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing_count,
    "Missing Percentage": missing_percentage
}).sort_values("Missing Values", ascending=False)

display(missing_summary)

missing_plot = missing_summary[missing_summary["Missing Values"] > 0]

if len(missing_plot) > 0:
    plt.figure(figsize=(10, 5))
    sns.barplot(
        x=missing_plot.index.astype(str),
        y=missing_plot["Missing Values"]
    )
    plt.title("Missing Values by Column")
    plt.xlabel("Column")
    plt.ylabel("Number of Missing Values")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("The dataset contains no missing values.")


## 6. Handling Missing Data


In [ ]:
df_clean = df.copy()

numeric_columns = df_clean.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df_clean.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

for col in numeric_columns:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_columns:
    if df_clean[col].isnull().any():
        mode_value = df_clean[col].mode()
        if not mode_value.empty:
            df_clean[col] = df_clean[col].fillna(mode_value.iloc[0])
        else:
            df_clean[col] = df_clean[col].fillna("Unknown")

print("Remaining missing values after handling:")
display(df_clean.isnull().sum().to_frame("Remaining Missing Values"))


## 7. Data Cleaning

The `rownames` column is an identifier rather than a meaningful analytical variable, so it is retained for reference but excluded from statistical and visualization analysis.

Column names are also converted into a consistent snake_case format.


In [ ]:
duplicates_before = df_clean.duplicated().sum()

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

duplicates_after = df_clean.duplicated().sum()

print("Duplicate rows before cleaning:", duplicates_before)
print("Duplicate rows after cleaning:", duplicates_after)

original_columns = df_clean.columns.tolist()

df_clean.columns = (
    df_clean.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("\nOriginal column names:")
print(original_columns)

print("\nCleaned column names:")
print(df_clean.columns.tolist())

print("\nCleaned dataset shape:", df_clean.shape)


## 8. Selecting Variables for Analysis


In [ ]:
# Identifier column is not used for EDA
id_column = "rownames"

analysis_df = df_clean.drop(columns=[id_column], errors="ignore").copy()

numeric_columns = analysis_df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = analysis_df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric variables used for analysis:")
print(numeric_columns)

print("\nCategorical variables used for analysis:")
print(categorical_columns)


## 9. Data Transformation

Standardization is applied to the meaningful numerical variables.

The identifier column is intentionally excluded because standardizing an ID does not provide useful analytical information.


In [ ]:
df_transformed = analysis_df.copy()

for col in numeric_columns:
    std = df_transformed[col].std()
    if std != 0 and not pd.isna(std):
        df_transformed[col + "_zscore"] = (
            (df_transformed[col] - df_transformed[col].mean()) / std
        )

print("Standardized variables created:")
print([col + "_zscore" for col in numeric_columns])

display(df_transformed.head())


# 10. Univariate Analysis

Univariate analysis examines one variable at a time.

Three visualizations are included:
1. Age distribution
2. Political knowledge distribution
3. Vote category distribution


### Visualization 1 – Distribution of Age


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=analysis_df, x="age", bins=15, kde=True)
plt.title("Univariate Analysis: Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


### Visualization 2 – Distribution of Political Knowledge


In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=analysis_df, x="political_knowledge")
plt.title("Univariate Analysis: Political Knowledge Distribution")
plt.xlabel("Political Knowledge")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


### Visualization 3 – Vote Category Distribution


In [ ]:
vote_counts = analysis_df["vote"].value_counts()

plt.figure(figsize=(9, 5))
sns.barplot(x=vote_counts.index, y=vote_counts.values)
plt.title("Univariate Analysis: Vote Category Distribution")
plt.xlabel("Vote Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

display(vote_counts.to_frame("Count"))


# 11. Bivariate Analysis

Bivariate analysis examines the relationship between two variables.

Three visualizations are included:
1. Age and political knowledge
2. Age distribution across vote categories
3. Vote category distribution by gender


### Visualization 1 – Age vs Political Knowledge


In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=analysis_df,
    x="age",
    y="political_knowledge",
    alpha=0.6
)
plt.title("Bivariate Analysis: Age vs Political Knowledge")
plt.xlabel("Age")
plt.ylabel("Political Knowledge")
plt.tight_layout()
plt.show()

correlation = analysis_df[["age", "political_knowledge"]].corr().iloc[0, 1]
print(f"Pearson correlation between age and political knowledge: {correlation:.4f}")


### Visualization 2 – Age by Vote Category


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(
    data=analysis_df,
    x="vote",
    y="age"
)
plt.title("Bivariate Analysis: Age by Vote Category")
plt.xlabel("Vote Category")
plt.ylabel("Age")
plt.tight_layout()
plt.show()


### Visualization 3 – Vote Category by Gender


In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(
    data=analysis_df,
    x="vote",
    hue="gender"
)
plt.title("Bivariate Analysis: Vote Category by Gender")
plt.xlabel("Vote Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


# 12. Multivariate Analysis

Multivariate analysis examines three or more variables simultaneously.

Three visualizations are included:
1. Correlation heatmap
2. Pair plot
3. Age vs political knowledge with gender as a third variable


### Visualization 1 – Correlation Heatmap


In [ ]:
correlation_columns = [
    "age",
    "economic_cond_national",
    "economic_cond_household",
    "blair",
    "hague",
    "kennedy",
    "europe",
    "political_knowledge"
]

correlation_matrix = analysis_df[correlation_columns].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    square=True
)
plt.title("Multivariate Analysis: Correlation Heatmap")
plt.tight_layout()
plt.show()


### Visualization 2 – Pair Plot


In [ ]:
pair_columns = [
    "age",
    "economic_cond_national",
    "economic_cond_household",
    "political_knowledge"
]

pair_df = analysis_df[pair_columns].dropna()

sns.pairplot(pair_df)
plt.suptitle("Multivariate Analysis: Pair Plot", y=1.02)
plt.show()


### Visualization 3 – Age, Political Knowledge and Gender


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=analysis_df,
    x="age",
    y="political_knowledge",
    hue="gender",
    style="vote",
    alpha=0.65
)

plt.title("Multivariate Analysis: Age, Political Knowledge, Gender and Vote")
plt.xlabel("Age")
plt.ylabel("Political Knowledge")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 13. Final Dataset Summary


In [ ]:
print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)

print("\nTotal remaining missing values:", df_clean.isnull().sum().sum())
print("Total remaining duplicate rows:", df_clean.duplicated().sum())

print("\nAnalysis numeric variables:")
print(numeric_columns)

print("\nAnalysis categorical variables:")
print(categorical_columns)


# 14. Conclusion

The BEPS dataset was successfully loaded and explored.

The Phase 1 requirements were completed through:
- Dataset loading and inspection
- Basic statistical analysis
- Missing-value identification and handling
- Duplicate checking and removal
- Column-name cleaning
- Appropriate data transformation
- Three univariate visualizations
- Three bivariate visualizations
- Three multivariate visualizations

The identifier column was excluded from analytical visualizations so that the EDA focuses on meaningful variables rather than row identifiers.
